In [1]:
# ============================================================================
# Notebook: 04_descriptives.ipynb
#
# Verification Channels and Remote Work — descriptives, diagnostics, ICC, sensitivity
#
# Produces the standard components a multilevel paper needs before the models:
#   - Table 1 (per-variable summary of each analytic sample)
#   - Missingness (how the complete-case sample was formed)
#   - Correlations + VIF (collinearity check)
#   - ICC (justifies the country level)
#   - Sensitivity of the H2 test to the min-country-N filter
#
# Saves everything to results/tables/. Runs from notebooks/; root is one up.
# ============================================================================


# %%
# ---------------------------------------------------------------------------
# Cell 1 | Setup
# ---------------------------------------------------------------------------
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
TAB_DIR = ROOT / "results" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))
from recode import recode, build_sample, build_robust_2025
from models import (descriptive_table, missingness_table, correlations_and_vif,
                    icc_country, sensitivity_country_filter)

pd.set_option("display.max_colwidth", None)
print("Setup complete.")


# %%
# ---------------------------------------------------------------------------
# Cell 2 | Load analytic samples (processed) and recoded frames (for missingness)
# ---------------------------------------------------------------------------
s24 = pd.read_csv(PROC_DIR / "analysis_2024.csv")
s25 = pd.read_csv(PROC_DIR / "analysis_2025.csv")

# Recoded (pre-filter) frames are needed for the missingness table.
df24 = pd.read_csv(RAW_DIR / "survey_2024.csv", low_memory=False)
df25 = pd.read_csv(RAW_DIR / "survey_2025.csv", low_memory=False)
r24 = recode(df24, 2024)
r25 = recode(df25, 2025)
print("Loaded samples:", len(s24), len(s25))


# %%
# ---------------------------------------------------------------------------
# Cell 3 | Table 1 — descriptive summary
# ---------------------------------------------------------------------------
tab1 = pd.concat([descriptive_table(s24, "2024"),
                  descriptive_table(s25, "2025")], ignore_index=True)
print(tab1.to_string(index=False))


# %%
# ---------------------------------------------------------------------------
# Cell 4 | Missingness (how the complete-case analytic sample was formed)
# ---------------------------------------------------------------------------
miss = pd.concat([missingness_table(r24, "2024"),
                  missingness_table(r25, "2025")], ignore_index=True)
print(miss.to_string(index=False))


# %%
# ---------------------------------------------------------------------------
# Cell 5 | Correlations + VIF (collinearity check)
# ---------------------------------------------------------------------------
corr24, vif24 = correlations_and_vif(s24, "2024")
corr25, vif25 = correlations_and_vif(s25, "2025")

print("Correlations (jobsat, ai_use, workexp_z):")
print(pd.concat([corr24, corr25], ignore_index=True).to_string(index=False))
print("\nVIF (design-matrix terms; rule of thumb: >5 warrants a look):")
vif = pd.concat([vif24, vif25], ignore_index=True)
print(vif.to_string(index=False))


# %%
# ---------------------------------------------------------------------------
# Cell 6 | ICC — variance share at the country level (justifies the 2nd level)
# ---------------------------------------------------------------------------
icc = pd.concat([icc_country(s24, "2024"),
                 icc_country(s25, "2025")], ignore_index=True)
print(icc.to_string(index=False))
print("\nGuide: ICC < 0.05 = weak clustering; multilevel still valid but modest.")


# %%
# ---------------------------------------------------------------------------
# Cell 7 | Sensitivity — does the H2 conclusion depend on the country filter?
# ---------------------------------------------------------------------------
sens = pd.concat([sensitivity_country_filter(r24, "2024"),
                  sensitivity_country_filter(r25, "2025")], ignore_index=True)
print("H2 interaction-only LRT across min-country-N thresholds:")
print(sens.to_string(index=False))
print("\nIf p stays > 0.05 across thresholds, the H2 null is robust to this choice.")


# %%
# ---------------------------------------------------------------------------
# Cell 8 | Save all tables
# ---------------------------------------------------------------------------
tab1.to_csv(TAB_DIR / "table1_descriptives.csv", index=False)
miss.to_csv(TAB_DIR / "missingness.csv", index=False)
pd.concat([corr24, corr25], ignore_index=True).to_csv(TAB_DIR / "correlations.csv", index=False)
vif.to_csv(TAB_DIR / "vif.csv", index=False)
icc.to_csv(TAB_DIR / "icc.csv", index=False)
sens.to_csv(TAB_DIR / "sensitivity_country_filter.csv", index=False)

print("Saved tables:")
for f in ["table1_descriptives.csv", "missingness.csv", "correlations.csv",
          "vif.csv", "icc.csv", "sensitivity_country_filter.csv"]:
    print("  results/tables/" + f)


# %%
# ---------------------------------------------------------------------------
# Cell 9 | Done
# ---------------------------------------------------------------------------
print("Descriptives, diagnostics, ICC, and sensitivity complete.")

Setup complete.
Loaded samples: 26576 16081
year                variable                                                                                                     stat
2024     N (analytic sample)                                                                                                    26576
2024     Countries (level-2)                                                                                                       48
2024 Job satisfaction (0-10)                                                                           M=6.95, SD=2.07, range=[0, 10]
2024 Work experience (years)                                                                          M=11.71, SD=9.27, range=[0, 50]
2024        AI use = Yes (%)                                                                                                    61.8%
2024        Work arrangement                                                              Hybrid 43.9%, Remote 38.7%, In-person 17.4%
2024              